# Vue des Derniers 10 messages par patient

In [0]:
%sql
CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_last_10_messages_per_patient AS

WITH ranked_records AS (
  SELECT 
    caseid,
    sbp,
    hr,
    spo2,
    temp,
    risk_level,
    alert,
    timestamp,
    shock_index,
    ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY timestamp DESC) AS rn
  FROM iotmlhealthcatalog.gold.vitaldbstream
)
SELECT 
  caseid,
  sbp,
  hr,
  spo2,
  temp,
  risk_level,
  alert,
  timestamp,
  shock_index
FROM ranked_records
WHERE rn <= 10
ORDER BY caseid, timestamp DESC

# Vue des Derniers 25 messages par patient

In [0]:
%sql
CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_last_25_messages_per_patient AS

WITH ranked_records AS (
  SELECT 
    caseid,
    sbp,
    hr,
    spo2,
    temp,
    risk_level,
    alert,
    timestamp,
    shock_index,
    ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY timestamp DESC) AS rn
  FROM iotmlhealthcatalog.gold.vitaldbstream
)
SELECT 
  caseid,
  sbp,
  hr,
  spo2,
  temp,
  risk_level,
  alert,
  timestamp,
  shock_index
FROM ranked_records
WHERE rn <= 25
ORDER BY caseid, timestamp DESC

# Vue des Dernières 10 minutes

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_last_10min_messages AS

WITH latest_timestamp AS (
    SELECT MAX(timestamp) AS timestamp_plus_recent
    FROM iotmlhealthcatalog.gold.vitaldbstream
)

SELECT
    v.caseid,
    v.risk_level,
    v.alert,
    v.timestamp,
    v.sbp,
    v.hr,
    v.spo2,
    v.temp,
    v.shock_index
FROM iotmlhealthcatalog.gold.vitaldbstream v
CROSS JOIN latest_timestamp l
WHERE v.timestamp BETWEEN
      l.timestamp_plus_recent - INTERVAL 600 SECONDS
      AND l.timestamp_plus_recent
ORDER BY v.timestamp DESC
;

# KPI — Patients actifs (10 dernières minutes)

Nombre de patients ayant envoyé au moins un message récemment.

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_kpi_active_patients AS

WITH latest_timestamp AS (
    SELECT MAX(timestamp) AS max_ts
    FROM iotmlhealthcatalog.gold.vitaldbstream
)

SELECT
    COUNT(DISTINCT caseid) AS active_patients
FROM iotmlhealthcatalog.gold.vitaldbstream v
CROSS JOIN latest_timestamp l
WHERE v.timestamp BETWEEN
      l.max_ts - INTERVAL 10 MINUTES
      AND l.max_ts
;

# KPI — Nombre de patients critiques

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_kpi_critical_patients AS

SELECT
    COUNT(DISTINCT caseid) AS critical_patients
FROM iotmlhealthcatalog.gold.vitaldbstream
WHERE
    risk_level = 'HIGH'
    OR alert != 'OK' 
;

# KPI — Dernier événement reçu

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_kpi_latest_event AS

SELECT
    MAX(timestamp) AS latest_event_timestamp
FROM iotmlhealthcatalog.gold.vitaldbstream
;

# KPI — Signes vitaux moyens globaux

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_kpi_avg_vitals AS

SELECT
    ROUND(AVG(sbp), 2) AS avg_sbp,
    ROUND(AVG(hr), 2) AS avg_hr,
    ROUND(AVG(spo2), 2) AS avg_spo2,
    ROUND(AVG(temp), 2) AS avg_temp,
    ROUND(AVG(shock_index), 2) AS avg_shock_index
FROM iotmlhealthcatalog.gold.vitaldbstream
;

# KPI — Messages reçus par minute

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_messages_per_minute AS

SELECT
    DATE_TRUNC('minute', timestamp) AS minute_window,
    COUNT(*) AS total_messages
FROM iotmlhealthcatalog.gold.vitaldbstream
GROUP BY DATE_TRUNC('minute', timestamp)
ORDER BY minute_window DESC
;

# KPI — Répartition des niveaux de risque

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_risk_distribution AS

SELECT
    risk_level,
    COUNT(*) AS total_records
FROM iotmlhealthcatalog.gold.vitaldbstream
GROUP BY risk_level
ORDER BY total_records DESC
;

# KPI — Derniers patients en alerte

In [0]:
%sql

CREATE OR REPLACE VIEW iotmlhealthcatalog.gold.vw_recent_alerts AS

SELECT
    caseid,
    timestamp,
    risk_level,
    alert,
    hr,
    spo2,
    sbp,
    shock_index
FROM iotmlhealthcatalog.gold.vitaldbstream
WHERE
    alert != 'OK'
    OR risk_level = 'HIGH'
ORDER BY timestamp DESC
LIMIT 20
;